In [35]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,image_path,number
0,images/image_0214.png,0
1,images/image_0187.png,4
2,images/image_0000.png,0
3,images/image_0262.png,5
4,images/image_0082.png,3


In [37]:
import torch

epochs = 6
batch_size = 32
learning_rate = 0.0002

device = torch.device("cuda")
device

device(type='cuda')

In [38]:
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image

train_transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor()
])

valid_transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor()
])

class MyDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.image_path.iloc[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        label = torch.tensor(self.df.number.iloc[idx], dtype=torch.int64).view(1)
        return img, label

from sklearn.model_selection import train_test_split

df_1, df_2 = train_test_split(df_train, test_size=0.2, random_state=42)

train_dataset = MyDataset(df_1, train_transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size)

test_dataset = MyDataset(df_2, valid_transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

train_dataset[0][0][0][0][0]

tensor(0.7451)

In [39]:
import torch.nn as nn
import torchvision

model = torchvision.models.resnet18(weights="IMAGENET1K_V1")
model.fc = nn.Sequential(
    nn.Linear(512, 256),
    nn.ReLU(),
    nn.Linear(256, 1)
)

model = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.L1Loss()

model

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [40]:
def train(model, criterion, train_loader, optimizer, epoch):
    model.train()
    total_loss = 0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        loss = criterion(model(inputs), labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print("Epoch ", epoch)
    print("Training Loss ", total_loss/len(train_loader))
def test(model, criterion, test_loader, epoch):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            loss = criterion(model(inputs), labels)
            total_loss += loss.item()
    print("Validation Loss ", total_loss/len(test_loader))

In [41]:
for epoch in range(1, epochs+1):
    train(model, criterion, train_loader, optimizer, epoch)
    test(model, criterion, test_loader, epoch)

Epoch  1
Training Loss  1.8458683252334596
Validation Loss  1.1345962285995483
Epoch  2
Training Loss  0.8720230221748352
Validation Loss  2.0324268341064453
Epoch  3
Training Loss  0.7454246520996094
Validation Loss  0.817563533782959
Epoch  4
Training Loss  0.4825121879577637
Validation Loss  0.8994836211204529
Epoch  5
Training Loss  0.4041612148284912
Validation Loss  0.7715374231338501
Epoch  6
Training Loss  0.36064006090164186
Validation Loss  0.9027251899242401


In [42]:
final_transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor()
])

class FinalDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        path = self.df.image_path.iloc[idx]
        img = Image.open(path).convert("RGB")
        img = self.transform(img)
        return img

final_dataset = FinalDataset(df_test, final_transform)
final_loader = DataLoader(final_dataset, batch_size=batch_size)

final_dataset[0][0][0][0]

tensor(0.3843)

In [43]:
import numpy as np

model.eval()
preds = []

with torch.no_grad():
    for inputs in final_loader:
        inputs = inputs.to(device)
        outputs = model(inputs).squeeze(1).cpu().numpy()
        outputs = np.round(outputs)
        outputs = np.clip(outputs, 0, 5)
        preds.extend(outputs)

preds[:10]

[3.0, 1.0, 3.0, 1.0, 5.0, 5.0, 3.0, 1.0, 4.0, 0.0]

In [44]:
df_preds = pd.DataFrame({
    "image_path": df_test.image_path,
    "number": preds
})

df_preds.head()

,image_path,number
0,images/image_0119.png,3.0
1,images/image_0227.png,1.0
2,images/image_0160.png,3.0
3,images/image_0091.png,1.0
4,images/image_0141.png,5.0


In [45]:
df_preds.number.value_counts()

number
3.0    30
4.0    21
2.0    20
1.0    13
5.0    12
0.0     4
Name: count, dtype: int64

In [46]:
df_preds.to_csv("submission.csv", index=False)